In [2]:
!pip install langchain langchain-community langchain-core langchain-text-splitters chromadb pypdf sentence-transformers

In [3]:
!pip install -U langchain-text-splitters

In [4]:
!pip install -U langchain-community langchain-google-genai langchain chromadb pypdf langgraph

In [1]:
!pip install --upgrade -q langchain-google-genai langchain-community chromadb

In [10]:
!pip install --upgrade -q langchain-google-genai langchain-community chromadb

In [7]:
from google.colab import files
uploaded = files.upload()

Saving RAG_Project_Documentation.pdf.pdf to RAG_Project_Documentation.pdf.pdf


In [8]:
# PDF load + split

In [15]:
import os
from typing import TypedDict, List
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# २. State
class AgentState(TypedDict):
    question: str
    answer: str
    context: List[str]
    needs_human: bool

# ३. Retrieval
def retrieve_docs(state: AgentState):
    print("--- LOG: STARTING RETRIEVAL ---")
    pdf_path = "RAG_Project_Documentation.pdf"
    if not os.path.exists(pdf_path):
        pdf_path = "RAG_Project_Documentation.pdf.pdf"

    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = text_splitter.split_documents(documents)

    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore = Chroma.from_documents(chunks, embeddings)
    return {"context": vectorstore.as_retriever(search_kwargs={"k": 2}).invoke(state["question"])}

# ४. Generation
def generate_answer(state: AgentState):
    print("--- LOG: GENERATING RESPONSE ---")
    if not state["context"]:
        return {"answer": "No info found.", "needs_human": True}


    context_data = state["context"][0].page_content[:500]
    final_ans = f"Based on the document: {context_data}"

    return {"answer": final_ans, "needs_human": False}

# ५. Graph Setup
from langgraph.graph import StateGraph, END
workflow = StateGraph(AgentState)
workflow.add_node("retriever", retrieve_docs)
workflow.add_node("generator", generate_answer)
workflow.set_entry_point("retriever")
workflow.add_edge("retriever", "generator")
workflow.add_edge("generator", END)
app = workflow.compile()

# ६. Run
result = app.invoke({"question": "What are the features?"})
print("\n" + "="*50)
print(f"ASSISTANT: {result['answer']}")
print(f"HUMAN ESCALATION: {result['needs_human']}")
print("="*50)

--- LOG: STARTING RETRIEVAL ---


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- LOG: GENERATING RESPONSE ---

ASSISTANT: Based on the document: Project Documentation: RAG-Based 
Customer Support Assistant 
1. High-Level Design (HLD) 
System Overview 
This project is a Retrieval-Augmented Generation (RAG) system designed for Customer 
Support. It allows users to query information from a specific PDF knowledge base. The system 
uses a graph-based workflow to ensure structured execution and decision -making. 
Architecture Components 
 User Interface: Command Line Interface (CLI) for user queries.
HUMAN ESCALATION: False


In [14]:
# Test

In [19]:
def ask(user_question: str):
    # This block automatically checks which variable name you used for the graph
    try:
        # Try using 'customer_support_app'
        response = customer_support_app.invoke({"question": user_question})
    except NameError:
        # If that fails, try using 'app'
        response = app.invoke({"question": user_question})

    return response['answer']

# --- RUN THE TEST AGAIN ---
if __name__ == "__main__":
    print("\n" + "="*60)
    print("       CUSTOMER SUPPORT AI AGENT - PRODUCTION TEST")
    print("="*60)

    # Try asking about the architecture instead of refund
query_final = "What are the architecture components?"
print(f"\n[USER]: {query_final}")
print(f"[AI]:   {ask(query_final)}")


       CUSTOMER SUPPORT AI AGENT - PRODUCTION TEST

[USER]: What are the architecture components?
--- LOG: STARTING RETRIEVAL ---


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- LOG: GENERATING RESPONSE ---
[AI]:   Based on the document:  Embedding Model: Google Generative AI Embeddings  (models/embedding-001). 
 Vector Database: ChromaDB for high-performance similarity search. 
 Orchestration: LangGraph to manage state and control flow.  
 LLM Engine: Gemini 1.5 Flash for contextual answer generation.  
Technology Stack 
Component Technology 
Programming Language Python 3.10+ 
GenAI API Google Gemini API 
Framework LangChain & LangGraph 
Vector Store ChromaDB 
 
2. Low-Level Design (LLD)
